# 🏪 Thực hành Sắp xếp và biến đổi dữ liệu

Trong notebook này, chúng ta sẽ luyện tập các kỹ thuật:
- Hierarchical Indexing
- Combining & Merging
- Concatenation
- Reshaping & Pivoting

Dữ liệu sử dụng: [Walmart Store Sales Forecasting](https://www.kaggle.com/competitions/walmart-recruiting-store-sales-forecasting)

---

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

# Đọc dữ liệu (ưu tiên data/; fallback sang data/walmart/)
base = Path('data')
train_path = base / 'train.csv'
features_path = base / 'features.csv'
stores_path = base / 'stores.csv'

if not train_path.exists():
    train_path = base / 'walmart' / 'train.csv'
if not features_path.exists():
    features_path = base / 'walmart' / 'features.csv'
if not stores_path.exists():
    stores_path = base / 'walmart' / 'stores.csv'

train = pd.read_csv(train_path)
features = pd.read_csv(features_path)
stores = pd.read_csv(stores_path)

train['Date'] = pd.to_datetime(train['Date'])
features['Date'] = pd.to_datetime(features['Date'])

print('Loaded:', train_path, features_path, stores_path)
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


# Phần 1: Chỉ số phân cấp
**Ngữ cảnh:** Dữ liệu doanh số theo `Store` và `Dept`. Nếu chỉ dùng index 1 cấp thì khó truy vấn chi tiết. MultiIndex sẽ hữu ích.

---

**Bài 1:** Tạo MultiIndex từ hai cột `Store` và `Dept`, sau đó hiển thị dữ liệu của `Store = 1, Dept = 1`. 

In [1]:
# Tạo MultiIndex từ Store và Dept
train_mi = train.set_index(['Store', 'Dept']).sort_index()

# Hiển thị dữ liệu của Store=1, Dept=1
store1_dept1 = train_mi.loc[(1, 1)]
print("Dữ liệu Store=1, Dept=1:")
print(store1_dept1.head())

**Bài 2:** Lấy toàn bộ dữ liệu của `Store = 2` (tất cả Dept).

In [3]:
# Lấy toàn bộ dữ liệu của Store=2 (tất cả Dept)
store2_all = train_mi.loc[2]
print("Dữ liệu Store=2 (tất cả Dept):")
print(store2_all.head())

**Bài 3:** Lấy dữ liệu của tất cả các `Dept = 5` trên toàn bộ các `Store`.

In [5]:
# Lấy dữ liệu Dept=5 trên toàn bộ các Store
dept5_all = train_mi.xs(5, level='Dept')
print("Dữ liệu Dept=5 trên toàn bộ Store:")
print(dept5_all.head())

**Bài 4:** Tính doanh số trung bình cho từng `Dept` trong mỗi `Store` bằng `groupby` rồi chuyển kết quả thành MultiIndex.

In [7]:
# Tính doanh số trung bình theo từng Dept trong mỗi Store
avg_sales_store_dept = (
    train.groupby(['Store', 'Dept'])['Weekly_Sales']
    .mean()
    .to_frame('Avg_Weekly_Sales')
)

print("Doanh số trung bình theo Store-Dept (MultiIndex):")
print(avg_sales_store_dept.head(10))

**Bài 5:** Sau khi tạo MultiIndex, thử dùng `.xs()` để truy cập nhanh một cấp chỉ số (ví dụ: Dept = 10 của tất cả các Store).

In [9]:
# Dùng xs để lấy Dept=10 của tất cả Store
dept10_all_stores = train_mi.xs(10, level='Dept')
print("Dữ liệu Dept=10 của tất cả Store:")
print(dept10_all_stores.head())

# Phần 2: Kết hợp các tập dữ liệu
**Ngữ cảnh:** Dữ liệu doanh số (`train.csv`) cần ghép với thông tin kinh tế khác (`features.csv`) để thấy được các yếu tố khác ảnh hưởng tới doanh số như thế nào.

---

**Bài 1:** Merge `train` với `features` dựa trên `Store` và `Date` (left join).

In [10]:
# Merge train với features theo Store và Date (left join)
train_features = train.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))

print("Kích thước sau merge train-features:", train_features.shape)
print(train_features.head())

**Bài 2:** Kiểm tra có bao nhiêu bản ghi trong `train` không tìm thấy trong `features`.

In [12]:
# Các cột thuộc features (trừ key)
feature_cols = [c for c in features.columns if c not in ['Store', 'Date']]

# Đếm bản ghi không match features: toàn bộ cột feature bị NaN
missing_in_features = train_features[feature_cols].isna().all(axis=1).sum()
print("Số bản ghi train không tìm thấy trong features:", missing_in_features)

**Bài 3:** Thêm dữ liệu `stores.csv` để có thêm cột `Type` và `Size` cho mỗi `Store`.

In [14]:
# Merge thêm stores để có Type và Size
full_data = train_features.merge(stores, on='Store', how='left')

print("Kích thước sau khi merge đủ 3 bảng:", full_data.shape)
print(full_data[['Store', 'Dept', 'Date', 'Weekly_Sales', 'Type', 'Size']].head())

**Bài 4:** Tìm `Store` nào có doanh số trung bình cao nhất khi merge đủ cả 3 bảng (`train`, `features`, `stores`).

In [15]:
# Doanh số trung bình theo Store
store_avg_sales = full_data.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)

best_store = store_avg_sales.index[0]
best_value = store_avg_sales.iloc[0]

print(f"Store có doanh số trung bình cao nhất: {best_store}")
print(f"Doanh số trung bình: {best_value:,.2f}")

**Bài 5:** Sau khi merge, tính doanh số trung bình của các `Store` theo từng loại `Type` (A/B/C).

In [17]:
# Doanh số trung bình theo Type (A/B/C)
type_avg_sales = full_data.groupby('Type')['Weekly_Sales'].mean().sort_values(ascending=False)

print("Doanh số trung bình theo loại Store:")
print(type_avg_sales)

**Bài 6:** Chia `train` thành 3 DataFrame theo năm (2010, 2011, 2012).

In [18]:
# Chia train thành 3 DataFrame theo năm
train_2010 = train[train['Date'].dt.year == 2010].copy()
train_2011 = train[train['Date'].dt.year == 2011].copy()
train_2012 = train[train['Date'].dt.year == 2012].copy()

print("Số dòng theo năm:")
print("2010:", len(train_2010))
print("2011:", len(train_2011))
print("2012:", len(train_2012))

**Bài 7:** Ghép chúng lại thành một DataFrame duy nhất bằng `pd.concat`.

In [20]:
# Ghép 3 DataFrame lại bằng concat theo hàng
train_concat = pd.concat([train_2010, train_2011, train_2012], axis=0, ignore_index=True)

print("Kích thước sau concat theo hàng:", train_concat.shape)
print(train_concat.head())

**Bài 8:** Thử ghép theo **cột** (`axis=1`) và quan sát sự khác biệt.

In [22]:
# Concat theo cột (axis=1)
concat_axis1 = pd.concat(
    [
        train_2010.reset_index(drop=True).add_prefix('2010_'),
        train_2011.reset_index(drop=True).add_prefix('2011_'),
        train_2012.reset_index(drop=True).add_prefix('2012_')
    ],
    axis=1
)

print("Kích thước concat theo cột:", concat_axis1.shape)
print(concat_axis1.head())

# Khác biệt: axis=0 nối thêm dòng; axis=1 nối thêm cột theo chỉ mục dòng.

**Bài 9:** Gán keys = `['2010','2011','2012']` khi concat để tạo MultiIndex ở cấp trên.

In [24]:
# Concat có keys để tạo MultiIndex ở cấp trên
concat_with_keys = pd.concat(
    [train_2010, train_2011, train_2012],
    keys=['2010', '2011', '2012'],
    names=['YearBlock']
)

print("Dữ liệu concat với keys (MultiIndex):")
print(concat_with_keys.head())
print("\nTên index levels:", concat_with_keys.index.names)

**Bài 10:** So sánh số bản ghi trước và sau khi concat để kiểm tra tính toàn vẹn dữ liệu.

In [25]:
# So sánh số bản ghi trước và sau concat
before_concat = len(train_2010) + len(train_2011) + len(train_2012)
after_concat = len(train_concat)

print("Số bản ghi trước concat:", before_concat)
print("Số bản ghi sau concat:", after_concat)
print("Toàn vẹn dữ liệu:", before_concat == after_concat)

# Phần 3: Tổ chức và Cấu trúc lại dữ liệu
**Ngữ cảnh:** Ban lãnh đạo cần báo cáo doanh số dễ đọc hơn, thay vì dữ liệu dạng long.

---

**Bài 1:** Dùng `pivot_table` để tính doanh số trung bình theo `Store` (hàng) và `Dept` (cột).

In [26]:
# Pivot table: doanh số trung bình theo Store (hàng) và Dept (cột)
pivot_store_dept = train.pivot_table(
    values='Weekly_Sales',
    index='Store',
    columns='Dept',
    aggfunc='mean'
)

print("Pivot doanh số trung bình theo Store x Dept:")
print(pivot_store_dept.head())

**Bài 2:** Lấy doanh số trung bình theo từng `Store` theo từng năm (sử dụng `Date`).

In [27]:
# Tạo cột Year từ Date rồi pivot theo Store x Year
train_with_year = train.copy()
train_with_year['Year'] = train_with_year['Date'].dt.year

pivot_store_year = train_with_year.pivot_table(
    values='Weekly_Sales',
    index='Store',
    columns='Year',
    aggfunc='mean'
)

print("Doanh số trung bình theo Store và năm:")
print(pivot_store_year.head())

**Bài 3:** Dùng `stack` và `unstack` để biến đổi bảng pivot thành long rồi wide trở lại.

In [28]:
# Stack: wide -> long
long_from_pivot = pivot_store_dept.stack(dropna=False).reset_index(name='Avg_Weekly_Sales')

# Unstack/pivot lại: long -> wide
wide_again = long_from_pivot.pivot(index='Store', columns='Dept', values='Avg_Weekly_Sales')

print("Dữ liệu sau stack (long):")
print(long_from_pivot.head())
print("\nDữ liệu sau unstack/pivot (wide lại):")
print(wide_again.head())

**Bài 4:** Tạo pivot table để xem doanh số trung bình theo `Type` của Store và theo từng năm.

In [30]:
# Merge train với stores để có Type
train_store = train.merge(stores[['Store', 'Type']], on='Store', how='left')
train_store['Year'] = train_store['Date'].dt.year

# Pivot theo Type và Year
pivot_type_year = train_store.pivot_table(
    values='Weekly_Sales',
    index='Type',
    columns='Year',
    aggfunc='mean'
)

print("Doanh số trung bình theo Type và năm:")
print(pivot_type_year)